<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ ScenA Audio Expressive Speech Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 GPU Edition — Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Zero-Shot Voice Cloning • Intent-Aware Multi-Speaker Dialogue Generation | Gemma-3-12B-it Text Encoder</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Google%20Colab-T4%20GPU-blue?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### Quick Start Guide
1. **Settings → Runtime type → T4 GPU** (Standard/Free tier GPU).
2. Run all cells in order.
3. Use the Gradio web interface to input your script and upload voice references.
4. *Note: We use the ungated community mirror `mlabonne/gemma-3-12b-it-abliterated` as the text encoder, so no Hugging Face tokens are required!*

In [ ]:
#@title 🖥️ Step 1: Environment Setup & Optimizer
# Define environment memory flags and verify GPU presence
import os, sys, gc, psutil, subprocess

print("=== Colab Environment Verification ===")
try:
    subprocess.run(["nvidia-smi"], check=True)
    print("✅ GPU is active and verified!")
except Exception:
    print("❌ WARNING: No GPU detected. Please go to Runtime -> Change runtime type and select T4 GPU.")

# Set memory management environment variables for T4 (15GB VRAM)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Run initial garbage collection
gc.collect()
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ GPU Memory optimized!")

In [ ]:
#@title 📦 Step 2: Install Dependencies & Clone Repo
# Install dependencies and setup the public ScenA repository
import sys, os, subprocess, shutil

print("=== Installing Dependencies ===")
# Note: Google Colab pre-installs PyTorch. Removing the slow torch reinstall (which takes 2-3 minutes
# and causes network/cell timeouts) makes this cell run in seconds and avoids package conflicts!
print("Installing required model loading and UI packages (transformers, accelerate, bitsandbytes, gradio)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.48.0", "accelerate>=0.28.0", "bitsandbytes>=0.42.0",
    "gradio>=4.0.0", "huggingface_hub", "safetensors", "av", "soundfile"
], check=True)

print("=== Cloning ScenA Repository ===")
# Ensure we clone cleanly by deleting any previous failed clone folders
if os.path.exists("scena"):
    print("Removing existing 'scena' directory for a clean clone...")
    try:
        shutil.rmtree("scena")
    except Exception as e:
        print(f"Warning: Failed to delete 'scena' directory via shutil: {e}. Trying shell rm force delete...")
        subprocess.run(["rm", "-rf", "scena"])

# Clone with depth 1 and skip large LFS weights (which we download separately in Step 4)
env = os.environ.copy()
env["GIT_LFS_SKIP_SMUDGE"] = "1"
try:
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/finmickey/scena.git"
    ], env=env, check=True)
    print("✅ Git clone completed successfully!")
except subprocess.CalledProcessError as e:
    print(f"❌ Error during git clone: {e}")
    raise e
gemma_path = "scena/packages/ltx-core/src/ltx_core/text_encoders/gemma/__init__.py"
if os.path.exists(gemma_path):
    print("✅ Verified: gemma package exists at target path!")
else:
    print("❌ Critical Verification Failure: gemma package was not found after clone!")
    print("Current working directory:", os.getcwd())
    print("Files in current directory:", os.listdir("."))
    if os.path.exists("scena"):
        print("Files inside 'scena' directory:", os.listdir("scena"))
    raise FileNotFoundError("Cloned repository is missing files or clone failed. Please run Step 2 again.")

# Add cloned repository packages to Python path dynamically
import glob
for pkg in ["ltx_core", "ltx_pipelines"]:
    matches = glob.glob(f"**/{pkg}", recursive=True)
    if matches:
        parent = os.path.dirname(os.path.abspath(matches[0]))
        if parent not in sys.path:
            sys.path.append(parent)
            print(f"Added package root to path: {parent}")
    else:
        fallback = os.path.abspath(f"scena/packages/{pkg.replace('_', '-')}")
        sys.path.append(fallback)
        print(f"Fallback added to path: {fallback}")

print("✅ Repository cloned and packages registered successfully!")

In [ ]:
#@title ⚙️ Step 3: Apply Memory Optimizations Patches
# Patch model code programmatically to run on Colab's 12.7GB CPU RAM and 15GB VRAM
import os, glob

# Dynamic search helper to find python files in the cloned directory
def find_file(pattern, filename):
    # Try searching from current directory first
    paths = glob.glob(pattern, recursive=True)
    if paths:
        return paths[0]
    # Try searching from /content/scena or /content/ recursively (standard Colab paths)
    for base in ["/content/scena", "/content", "."]:
        if os.path.exists(base):
            search_pattern = os.path.join(base, pattern)
            paths = glob.glob(search_pattern, recursive=True)
            if paths:
                return paths[0]
    # Fallback to standard path if glob fails
    fallback = os.path.join("scena", filename)
    if os.path.exists(fallback):
        return fallback
    fallback_abs = os.path.join("/content/scena", filename)
    if os.path.exists(fallback_abs):
        return fallback_abs
    raise FileNotFoundError(f"Could not locate {filename} inside the workspace. Please check if Step 2 completed successfully.")

print("Locating files to patch...")
try:
    blocks_py = find_file("**/utils/blocks.py", "packages/ltx-pipelines/src/ltx_pipelines/utils/blocks.py")
    gpu_model_py = find_file("**/utils/gpu_model.py", "packages/ltx-pipelines/src/ltx_pipelines/utils/gpu_model.py")
    t2aud_ref_cond_py = find_file("**/t2aud_ref_cond.py", "packages/ltx-pipelines/src/ltx_pipelines/t2aud_ref_cond.py")
    feature_extractor_py = find_file("**/gemma/feature_extractor.py", "packages/ltx-core/src/ltx_core/text_encoders/gemma/feature_extractor.py")
    registry_py = find_file("**/loader/registry.py", "packages/ltx-core/src/ltx_core/loader/registry.py")
    print(f"  Found blocks.py: {blocks_py}")
    print(f"  Found gpu_model.py: {gpu_model_py}")
    print(f"  Found t2aud_ref_cond.py: {t2aud_ref_cond_py}")
    print(f"  Found feature_extractor.py: {feature_extractor_py}")
    print(f"  Found registry.py: {registry_py}")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    raise e

print("Patching blocks.py to load Gemma-3-12B-it in 4-bit...")
with open(blocks_py, "r") as f:
    blocks_code = f.read()

target_blocks = """    def _build_text_encoder(self) -> torch.nn.Module:
        \"\"\"Build the Gemma text encoder (non-streaming path).\"\"\"
        return self._text_encoder_builder.build(device=self._device, dtype=self._dtype).eval()"""

replacement_blocks = """    def _build_text_encoder(self) -> torch.nn.Module:
        \"\"\"Build the Gemma text encoder (non-streaming path).\"\"\"
        # Load Gemma-3-12B-it pre-quantized 4-bit directly using Hugging Face to save VRAM and download time!
        from transformers import Gemma3ForConditionalGeneration, Gemma3Processor
        from ltx_core.text_encoders.gemma.encoders.base_encoder import GemmaTextEncoder
        from ltx_core.text_encoders.gemma.tokenizer import LTXVGemmaTokenizer
        import torch

        print("Loading pre-quantized Gemma-3-12B-it 4-bit...")
        model = Gemma3ForConditionalGeneration.from_pretrained(
            self._gemma_root,
            torch_dtype=self._dtype,
            device_map="auto"
        )

        tokenizer = LTXVGemmaTokenizer(self._gemma_root, 1024)
        processor = Gemma3Processor.from_pretrained(self._gemma_root)

        encoder = GemmaTextEncoder(
            model=model,
            tokenizer=tokenizer,
            processor=processor,
            dtype=self._dtype
        )
        return encoder.eval()"""

if target_blocks in blocks_code:
    blocks_code = blocks_code.replace(target_blocks, replacement_blocks)
else:
    print("  ❌ Error: target_blocks string not found in blocks.py!")

target_blocks_dt = """        builder = self._transformer_builder.with_module_ops(module_ops).with_sd_ops(sd_ops).with_loras(loras)
        if self._quantization is not None:
            builder = builder.with_fuse_rule(self._quantization.fuse_rule)
        return X0Model(builder.build(device=target, **kwargs)).to(target).eval()"""

replacement_blocks_dt = """        builder = self._transformer_builder.with_module_ops(module_ops).with_sd_ops(sd_ops).with_loras(loras)
        if self._quantization is not None:
            builder = builder.with_fuse_rule(self._quantization.fuse_rule)
        return X0Model(builder.build(device=target, dtype=self._dtype, **kwargs)).to(target).eval()"""

if target_blocks_dt in blocks_code:
    blocks_code = blocks_code.replace(target_blocks_dt, replacement_blocks_dt)
else:
    print("  ❌ Error: target_blocks_dt string not found in blocks.py!")

target_prompt_call = """    def __call__(
        self,
        prompts: list[str],
        *,
        enhance_first_prompt: bool = False,
        enhance_prompt_image: str | None = None,
        enhance_prompt_seed: int = 42,
    ) -> list[EmbeddingsProcessorOutput]:
        \"\"\"Encode *prompts* through Gemma -> embeddings processor, freeing each model after use.\"\"\"
        logger.info("Building text encoder from %s", self._gemma_root)
        with self._text_encoder_ctx() as text_encoder:
            if enhance_first_prompt:
                prompts = list(prompts)
                prompts[0] = generate_enhanced_prompt(
                    text_encoder, prompts[0], enhance_prompt_image, seed=enhance_prompt_seed
                )
            raw_outputs = text_encoder.encode(prompts)
        logger.info("Text encoder done, building embeddings processor from %s", self._checkpoint_path)

        with gpu_model(self._build_embeddings_processor()) as embeddings_processor:
            result = [embeddings_processor.process_hidden_states(hs, mask) for hs, mask in raw_outputs]
        logger.info("Prompt encoding complete")
        return result"""

replacement_prompt_call = """    def __call__(
        self,
        prompts: list[str],
        *,
        enhance_first_prompt: bool = False,
        enhance_prompt_image: str | None = None,
        enhance_prompt_seed: int = 42,
    ) -> list[EmbeddingsProcessorOutput]:
        \"\"\"Encode *prompts* through Gemma -> embeddings processor, freeing each model after use.\"\"\"
        logger.info("Building text encoder from %s", self._gemma_root)
        with self._text_encoder_ctx() as text_encoder:
            if enhance_first_prompt:
                prompts = list(prompts)
                prompts[0] = generate_enhanced_prompt(
                    text_encoder, prompts[0], enhance_prompt_image, seed=enhance_prompt_seed
                )
            raw_outputs = text_encoder.encode(prompts)

        # Explicitly delete text encoder and clear CUDA cache immediately to release VRAM
        import gc
        import torch
        del text_encoder
        gc.collect()
        torch.cuda.empty_cache()

        logger.info("Text encoder done, building embeddings processor from %s", self._checkpoint_path)

        with gpu_model(self._build_embeddings_processor()) as embeddings_processor:
            result = [embeddings_processor.process_hidden_states(hs, mask) for hs, mask in raw_outputs]

        # Explicitly delete embeddings processor and clear CUDA cache immediately to release VRAM
        del embeddings_processor
        gc.collect()
        torch.cuda.empty_cache()

        logger.info("Prompt encoding complete")
        return result"""

if target_prompt_call in blocks_code:
    blocks_code = blocks_code.replace(target_prompt_call, replacement_prompt_call)
    print("  ✓ blocks.py PromptEncoder.__call__ successfully patched!")
else:
    print("  ❌ Error: target_prompt_call string not found in blocks.py!")

with open(blocks_py, "w") as f:
    f.write(blocks_code)
print("  ✓ ltx-pipelines/utils/blocks.py successfully patched!")

print("Patching gpu_model.py to support quantized models...")
with open(gpu_model_py, "r") as f:
    gpu_model_code = f.read()

target_gpu_model = """    finally:
        torch.cuda.synchronize()
        # .to("meta") releases storage for all parameters/buffers regardless
        # of their original device (CUDA or CPU).
        model.to("meta")
        cleanup_memory()"""

replacement_gpu_model = """    finally:
        torch.cuda.synchronize()
        # .to("meta") releases storage for all parameters/buffers regardless
        # of their original device (CUDA or CPU).
        try:
            model.to("meta")
        except Exception:
            try:
                model.to("cpu")
            except Exception:
                pass
        cleanup_memory()"""

if target_gpu_model in gpu_model_code:
    gpu_model_code = gpu_model_code.replace(target_gpu_model, replacement_gpu_model)
    with open(gpu_model_py, "w") as f:
        f.write(gpu_model_code)
    print("  ✓ ltx-pipelines/utils/gpu_model.py successfully patched!")
else:
    print("  ❌ Error: target_gpu_model string not found in gpu_model.py!")

print("Patching t2aud_ref_cond.py for dynamic hardware-accelerated float16 and shared weight caching...")
with open(t2aud_ref_cond_py, "r") as f:
    t2aud_code = f.read()

target_t2aud_imports = """from ltx_core.pipeline.ref_conds import EncodedRefCond"""
replacement_t2aud_imports = """from ltx_core.pipeline.ref_conds import EncodedRefCond
from ltx_core.loader.registry import Registry"""

if target_t2aud_imports in t2aud_code:
    t2aud_code = t2aud_code.replace(target_t2aud_imports, replacement_t2aud_imports)

target_t2aud = """    def __init__(
        self,
        checkpoint_path: str,
        gemma_root: str,
        audio_vae_path: str | None = None,
        max_ref_conds: int = 3,
        loras: list[LoraPathStrengthAndSDOps] | None = None,
        device: torch.device | None = None,
    ) -> None:
        self.dtype = torch.bfloat16
        self.device = device or get_device()
        self.max_ref_conds = max_ref_conds
        self._scheduler = LTX2Scheduler()
        # The audio VAE + vocoder live in the bundled audio_vae.safetensors; fall back to
        # the main checkpoint if a combined file is provided.
        audio_vae_path = audio_vae_path or checkpoint_path

        self.prompt_encoder = PromptEncoder(
            checkpoint_path=checkpoint_path,
            gemma_root=gemma_root,
            dtype=self.dtype,
            device=self.device,
            embeddings_processor_configurator=ScenaAudioOnlyEmbeddingsProcessorConfigurator,
            embeddings_processor_sd_ops=SCENA_AUDIO_ONLY_EMBEDDINGS_PROCESSOR_KEY_OPS,
        )
        self.stage = DiffusionStage(
            checkpoint_path=checkpoint_path,
            dtype=self.dtype,
            device=self.device,
            loras=tuple(loras or []),
            model_configurator=LTXAudioOnlyModelConfigurator,
            model_sd_ops=LTXV_AUDIO_ONLY_RENAMING_MAP,
        )
        self.audio_conditioner = AudioConditioner(checkpoint_path=audio_vae_path, dtype=self.dtype, device=self.device)
        self.audio_decoder = AudioDecoder(checkpoint_path=audio_vae_path, dtype=self.dtype, device=self.device)"""

replacement_t2aud = """    def __init__(
        self,
        checkpoint_path: str,
        gemma_root: str,
        audio_vae_path: str | None = None,
        max_ref_conds: int = 3,
        loras: list[LoraPathStrengthAndSDOps] | None = None,
        device: torch.device | None = None,
        registry: Registry | None = None,
    ) -> None:
        # Use native float16 on T4 / older GPUs (< SM 8.0) for hardware acceleration, otherwise bfloat16
        import torch
        if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
            self.dtype = torch.float16
            print("  ✓ CUDA SM < 8.0 (T4 GPU) detected. Switching pipeline precision to native float16!")
        else:
            self.dtype = torch.bfloat16
            print("  ✓ CUDA SM >= 8.0 detected. Using standard bfloat16 precision!")
        self.device = device or get_device()
        self.max_ref_conds = max_ref_conds
        self._scheduler = LTX2Scheduler()
        # The audio VAE + vocoder live in the bundled audio_vae.safetensors; fall back to
        # the main checkpoint if a combined file is provided.
        audio_vae_path = audio_vae_path or checkpoint_path

        self.prompt_encoder = PromptEncoder(
            checkpoint_path=checkpoint_path,
            gemma_root=gemma_root,
            dtype=self.dtype,
            device=self.device,
            registry=registry,
            embeddings_processor_configurator=ScenaAudioOnlyEmbeddingsProcessorConfigurator,
            embeddings_processor_sd_ops=SCENA_AUDIO_ONLY_EMBEDDINGS_PROCESSOR_KEY_OPS,
        )
        self.stage = DiffusionStage(
            checkpoint_path=checkpoint_path,
            dtype=self.dtype,
            device=self.device,
            loras=tuple(loras or []),
            registry=None,
            model_configurator=LTXAudioOnlyModelConfigurator,
            model_sd_ops=LTXV_AUDIO_ONLY_RENAMING_MAP,
        )
        self.audio_conditioner = AudioConditioner(checkpoint_path=audio_vae_path, dtype=self.dtype, device=self.device, registry=registry)
        self.audio_decoder = AudioDecoder(checkpoint_path=audio_vae_path, dtype=self.dtype, device=self.device, registry=registry)"""

if target_t2aud in t2aud_code:
    t2aud_code = t2aud_code.replace(target_t2aud, replacement_t2aud)
    with open(t2aud_ref_cond_py, "w") as f:
        f.write(t2aud_code)
    print("  ✓ t2aud_ref_cond.py successfully patched!")
else:
    print("  ❌ Error: target_t2aud string not found in t2aud_ref_cond.py!")

print("Patching feature_extractor.py to cast tokens to aggregate weights dtype...")
with open(feature_extractor_py, "r") as f:
    fe_code = f.read()

target_fe1 = """        encoded = torch.stack(hidden_states, dim=-1) if isinstance(hidden_states, (list, tuple)) else hidden_states
        dtype = encoded.dtype
        normed = _norm_and_concat_padded_batch(encoded, attention_mask)
        features = self.aggregate_embed(normed.to(dtype))"""

replacement_fe1 = """        encoded = torch.stack(hidden_states, dim=-1) if isinstance(hidden_states, (list, tuple)) else hidden_states
        # Cast to aggregate weight dtype to prevent Half vs BFloat16 matrix multiplication conflicts
        target_dtype = self.aggregate_embed.weight.dtype
        normed = _norm_and_concat_padded_batch(encoded, attention_mask)
        features = self.aggregate_embed(normed.to(target_dtype))"""

target_fe2 = """        encoded = torch.stack(hidden_states, dim=-1) if isinstance(hidden_states, (list, tuple)) else hidden_states
        normed = norm_and_concat_per_token_rms(encoded, attention_mask)
        normed = normed.to(encoded.dtype)"""

replacement_fe2 = """        encoded = torch.stack(hidden_states, dim=-1) if isinstance(hidden_states, (list, tuple)) else hidden_states
        normed = norm_and_concat_per_token_rms(encoded, attention_mask)
        # Cast to aggregate weight dtype to prevent Half vs BFloat16 matrix multiplication conflicts
        target_dtype = self.video_aggregate_embed.weight.dtype
        normed = normed.to(target_dtype)"""

if target_fe1 in fe_code:
    fe_code = fe_code.replace(target_fe1, replacement_fe1)
if target_fe2 in fe_code:
    fe_code = fe_code.replace(target_fe2, replacement_fe2)

with open(feature_extractor_py, "w") as f:
    f.write(fe_code)
print("  ✓ ltx-core/text_encoders/gemma/feature_extractor.py successfully patched!")

print("Patching registry.py to force cached state dict tensors to CPU...")
with open(registry_py, "r") as f:
    registry_code = f.read()

target_registry = """    def add(self, paths: list[str], sd_ops: SDOps | None, state_dict: StateDict) -> str:
        sd_id = self._generate_id(paths, sd_ops)
        with self._lock:
            if sd_id in self._state_dicts:
                raise ValueError(f"State dict retrieved from {paths} with {sd_ops} already added, check with get first")
            self._state_dicts[sd_id] = state_dict
        return sd_id"""

replacement_registry = """    def add(self, paths: list[str], sd_ops: SDOps | None, state_dict: StateDict) -> str:
        import torch
        from ltx_core.loader.primitives import StateDict
        # Move all tensors in state_dict.sd dict to CPU to save GPU VRAM
        cpu_sd = {k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in state_dict.sd.items()}
        # Rebuild custom StateDict class pointing to CPU
        new_state_dict = StateDict(
            sd=cpu_sd,
            device=torch.device("cpu"),
            size=state_dict.size,
            dtype=state_dict.dtype
        )
        sd_id = self._generate_id(paths, sd_ops)
        with self._lock:
            if sd_id in self._state_dicts:
                # Bypass the error to support dynamic pipeline reinstantiation safely
                self._state_dicts[sd_id] = new_state_dict
                return sd_id
            self._state_dicts[sd_id] = new_state_dict
        return sd_id"""

if target_registry in registry_code:
    registry_code = registry_code.replace(target_registry, replacement_registry)
    with open(registry_py, "w") as f:
        f.write(registry_code)
    print("  ✓ ltx-core/loader/registry.py successfully patched!")
else:
    print("  ❌ Error: target_registry string not found in registry.py!")

print("✅ Optimization patches successfully applied!")

In [ ]:
#@title 📥 Step 4: Download Model Checkpoints
# Download the ScenA model weights and the pre-quantized ungated Gemma-3-12B-it 4-bit mirror
import os
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("checkpoints", exist_ok=True)

print("Downloading ScenA checkpoint weights from Hugging Face...")
try:
    # Download ScenA flow-matching transformer weights
    hf_hub_download(
        repo_id="mifinkelson/scena",
        filename="scena.safetensors",
        local_dir="./checkpoints"
    )
    # Download Audio VAE and Vocoder weights
    hf_hub_download(
        repo_id="mifinkelson/scena",
        filename="audio_vae.safetensors",
        local_dir="./checkpoints"
    )
    print("✅ ScenA checkpoints downloaded successfully!")
except Exception as e:
    print(f"❌ Error downloading ScenA checkpoints: {e}")
    raise e

print("Downloading pre-quantized ungated Gemma-3-12B-it 4-bit mirror (Unsloth)...")
try:
    # Download the public, ungated Gemma-3-12b 4-bit mirror (only ~8.5GB download size!)
    snapshot_download(
        repo_id="unsloth/gemma-3-12b-it-bnb-4bit",
        local_dir="./gemma-3-12b-it-bnb-4bit"
    )
    print("✅ Gemma-3-12B 4-bit text encoder downloaded successfully!")
except Exception as e:
    print(f"❌ Error downloading Gemma model: {e}")
    raise e

print("✅ All checkpoints downloaded successfully!")

In [ ]:
#@title 🎙️ Step 5: Initialize Pipeline & Launch UI
# Set up pipeline inference and start the Gradio web interface
import gc, sys, os
import torch
import gradio as gr

# Ensure custom package folders are in the Python search path dynamically
import glob
for pkg in ["ltx_core", "ltx_pipelines"]:
    matches = glob.glob(f"**/{pkg}", recursive=True)
    if matches:
        parent = os.path.dirname(os.path.abspath(matches[0]))
        if parent not in sys.path:
            sys.path.append(parent)
            print(f"Added package root to path: {parent}")
    else:
        fallback = os.path.abspath(f"scena/packages/{pkg.replace('_', '-')}")
        sys.path.append(fallback)
        print(f"Fallback added to path: {fallback}")

from ltx_pipelines.t2aud_ref_cond import T2AudRefCondPipeline

# Global pipeline initialization (lazy loading)
pipe = None

def init_pipeline():
    global pipe
    if pipe is None:
        print("Initializing ScenA Pipeline... (Loading quantized weights)")
        from ltx_core.loader.registry import StateDictRegistry
        shared_registry = StateDictRegistry()
        pipe = T2AudRefCondPipeline(
            checkpoint_path="checkpoints/scena.safetensors",
            audio_vae_path="checkpoints/audio_vae.safetensors",
            gemma_root="./gemma-3-12b-it-bnb-4bit",
            device=torch.device("cuda"),
            max_ref_conds=4,
            registry=shared_registry
        )
        print("✅ Pipeline loaded successfully!")
    return pipe

def generate_scene(
    prompt,
    ref1, ref2, ref3, ref4,
    duration,
    num_inference_steps,
    guidance_scale,
    seed
):
    try:
        # Load the pipeline
        pipeline = init_pipeline()

        # Build reference audio list
        ref_audios = []
        for ref in [ref1, ref2, ref3, ref4]:
            if ref is not None:
                ref_audios.append(ref)

        if len(ref_audios) == 0:
            return None, "❌ Error: Please upload at least one voice reference clip."

        # Swap reference 1 and reference 2 to align Voice Reference 1 with "reference 1" in text
        if len(ref_audios) >= 2:
            ref_audios[0], ref_audios[1] = ref_audios[1], ref_audios[0]

        print(f"Generating scene with prompt: {prompt}")
        print(f"References loaded: {len(ref_audios)} (swapped for alignment), Duration: {duration}s")

        # Clean VRAM cache before generation
        gc.collect()
        torch.cuda.empty_cache()

        # Run inference (generate random seed if auto-seed is selected)
        if seed == -1:
            import random
            seed_val = random.randint(0, 2**32 - 1)
            print(f"Auto-seed selected. Generated random seed: {seed_val}")
        else:
            seed_val = int(seed)
        from ltx_core.components.guiders import MultiModalGuiderParams
        guider_params = MultiModalGuiderParams(cfg_scale=float(guidance_scale))

        audio_result = pipeline(
            prompt=prompt,
            ref_audio_paths=ref_audios,
            duration=float(duration),
            num_inference_steps=int(num_inference_steps),
            audio_guider_params=guider_params,
            seed=seed_val
        )

        # Save output file
        out_path = "output_scene.wav"
        audio_result.save(out_path)

        # Clean VRAM cache after generation
        gc.collect()
        torch.cuda.empty_cache()

        return out_path, "✅ Audio scene generated successfully!"

    except Exception as e:
        import traceback
        error_msg = f"❌ Error during generation:\n{str(e)}\n{traceback.format_exc()}"
        print(error_msg)
        return None, error_msg

# Custom Gradio design system
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2.2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.9); font-size: 1.1em; margin: 0; font-weight: 400; }
.generate-btn { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: bold !important; border: none !important; }
.generate-btn:hover { opacity: 0.9 !important; }
"""

with gr.Blocks(css=CSS) as demo:
    # Header Section
    gr.HTML(
        f"""
        <div class="brand-header">
            <h1 class="brand-title">🎙️ ScenA Audio Expressive Speech Generator</h1>
            <p class="brand-subtitle">Zero-Shot Voice Cloning & Ambient Scene Builder · AIQUEST Academy Edition</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            # Prompt inputs
            prompt_input = gr.Textbox(
                label="Prompt / Scene Script",
                placeholder='A farm at sunrise. The speaker from reference 1 says: "Way too early for this." The speaker from reference 2 chuckles: "Welcome to country life."',
                lines=5
            )

            # Voice reference uploads
            with gr.Group():
                gr.Markdown("### 🎤 Voice References (Upload 1 to 4 clean mono clips)")
                ref_1 = gr.Audio(label="Voice Reference 1", type="filepath")
                ref_2 = gr.Audio(label="Voice Reference 2", type="filepath")
                ref_3 = gr.Audio(label="Voice Reference 3", type="filepath")
                ref_4 = gr.Audio(label="Voice Reference 4", type="filepath")

            # Advanced generation controls
            with gr.Accordion("⚙️ Inference Controls", open=False):
                duration_slider = gr.Slider(
                    label="Audio Duration (seconds)", minimum=1.0, maximum=20.0, value=8.0, step=0.5
                )
                steps_slider = gr.Slider(
                    label="Inference Steps", minimum=10, maximum=100, value=30, step=5
                )
                guidance_slider = gr.Slider(
                    label="Guidance Scale", minimum=1.0, maximum=15.0, value=7.0, step=0.5
                )
                seed_input = gr.Number(
                    label="Random Seed (-1 for random)", value=-1, precision=0
                )

            btn = gr.Button("🎵 Generate Audio Scene", elem_classes="generate-btn")

        with gr.Column(scale=1):
            # Output player and status logging
            output_audio = gr.Audio(label="Generated Audio Scene", type="filepath")
            status_text = gr.Textbox(label="System Status / Logs", interactive=False)

            # Guidelines and tips
            gr.Markdown(
                """
                ### 💡 Prompting & Setup Guide
                *   **Referencing voices**: Refer to your speakers as `"the speaker from reference 1"`, `"reference 2"`, etc. in your prompt text.
                *   **Speech formatting**: Always place spoken words inside **double quotes** (e.g. ` कहता है: "नमस्कार" `).
                *   **Sound effects & ambience**: Describe background noises and sound effects in plain prose without quotes (e.g. `rain falls heavily on a tin roof`, `dogs barking in the distance`).
                *   **References**: Upload clean, single-speaker audio clips (mono format, under 20 seconds works best).
                """
            )

    btn.click(
        fn=generate_scene,
        inputs=[
            prompt_input,
            ref_1, ref_2, ref_3, ref_4,
            duration_slider,
            steps_slider,
            guidance_slider,
            seed_input
        ],
        outputs=[output_audio, status_text]
    )

    # Footer Section
    gr.HTML(
        """
        <div style="text-align: center; margin-top: 30px; border-top: 1px solid #ddd; padding-top: 20px;">
            <div align="center">
                <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank">
                    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
                </a>
                &nbsp;
                <a href="https://x.com/aiquestacademy" target="_blank">
                    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
                </a>
                &nbsp;
                <a href="https://aiquest.site" target="_blank">
                    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
                </a>
            </div>
            <p style="color:#6b7280; font-size:12px; margin-top:8px;">
                ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
            </p>
        </div>
        """
    )

# Launch the app
demo.queue().launch(share=True, debug=True)